# Task 3

In [9]:
import nest_asyncio
nest_asyncio.apply()

%run task_3_toolnode_example.py

LangGraph Manual Tool Calling - Persistent Multi-Turn Conversation

This system uses manual tool calling with ToolNode:
  - call_model: Invokes LLM with tool bindings
  - ToolNode: Executes requested tools in parallel
  - Loop: tools -> call_model (until no more tools needed)
  - Single persistent conversation across all turns
  - History managed automatically (trimmed after 100 messages)

Commands:
  - Type 'quit' or 'exit' to end the conversation
  - Type 'verbose' to enable detailed tracing
  - Type 'quiet' to disable detailed tracing

Available tools:
  - get_weather(location): Get weather information
  - get_population(city): Get population data
  - calculate(expression): Evaluate math expressions
[SYSTEM] Conversation graph created successfully (using manual ToolNode)
[SYSTEM] Graph visualization saved to 'langchain_manual_tool_graph.png'

[SYSTEM] Starting conversation...


NODE: input_node
[DEBUG] User input: hi
[DEBUG] Routing to call_model

NODE: call_model
[DEBUG] Calling mo

In [10]:
import nest_asyncio
nest_asyncio.apply()

%run task_3_react_agent_example.py

LangGraph ReAct Agent - Persistent Multi-Turn Conversation

This system uses create_react_agent with graph-based looping:
  - Single persistent conversation across all turns
  - History managed automatically (trimmed after 100 messages)
  - Loops via graph edges (no Python loops or checkpointing)

Commands:
  - Type 'quit' or 'exit' to end the conversation
  - Type 'verbose' to enable detailed tracing
  - Type 'quiet' to disable detailed tracing

Available tools:
  - get_weather(location): Get weather information
  - get_population(city): Get population data
  - calculate(expression): Evaluate math expressions
[SYSTEM] ReAct agent created successfully


/Users/wyd2hu/Library/CloudStorage/OneDrive-Personal/AI_Agent_Course/Topic 4/task_3_react_agent_example.py:375: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  react_agent = create_react_agent(


[SYSTEM] ReAct agent graph saved to 'langchain_react_agent.png'
[SYSTEM] Conversation graph saved to 'langchain_conversation_graph.png'

[SYSTEM] Starting conversation...


NODE: input_node
[DEBUG] User input: hey
[DEBUG] Routing to call_react_agent

NODE: call_react_agent
[DEBUG] Invoking ReAct agent with 1 messages in history
[DEBUG] Agent generated 1 new messages
[DEBUG] Response preview: Hello! How can I assist you today?...

NODE: output_node

Assistant: Hello! How can I assist you today?

NODE: input_node
[DEBUG] Exit command received
[DEBUG] Routing to END (exit requested)

[SYSTEM] Conversation ended. Goodbye!



# Task 5

In [ ]:
####################### NOTE #######################
## Testing tool in isolation

from youtube_transcript_api import YouTubeTranscriptApi

def get_youtube_transcript(video_id: str) -> str:
    """Fetch the transcript of a YouTube video by video ID."""
    transcript = YouTubeTranscriptApi().fetch(video_id)
    return " ".join([entry.text for entry in transcript])

print(get_youtube_transcript("dQw4w9WgXcQ")[:500])  # Print the first 500 characters of the transcript

[♪♪♪] ♪ We're no strangers to love ♪ ♪ You know the rules
and so do I ♪ ♪ A full commitment's
what I'm thinking of ♪ ♪ You wouldn't get this
from any other guy ♪ ♪ I just wanna tell you
how I'm feeling ♪ ♪ Gotta make you understand ♪ ♪ Never gonna give you up ♪ ♪ Never gonna let you down ♪ ♪ Never gonna run around
and desert you ♪ ♪ Never gonna make you cry ♪ ♪ Never gonna say goodbye ♪ ♪ Never gonna tell a lie
and hurt you ♪ ♪ We've known each other
for so long ♪ ♪ Your heart's been aching
but 


In [15]:
####################### NOTE #######################
# As can be seen from the output, the transcript is successfully fetched using the tool (printed "Fetching transcript for video ID ...."). 
# This confirms that the tool is also working correctly after being integrated into the React agent pipeline.
# Besides, here, I forced the model to use the tool by giving it a video link through explicit instructions in the system prompt.

import re
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent


@tool
def get_youtube_transcript(video_id: str) -> str:
    """Fetch the transcript of a YouTube video by video ID."""
    print(f"Fetching transcript for video ID: {video_id}")
    transcript = YouTubeTranscriptApi().fetch(video_id)
    return " ".join([entry.text for entry in transcript])


# Define the structured system prompt to enforce the pipeline
system_message = """You are an Educational Video Analyzer. 
When a user provides a video link, you must execute the following pipeline:
1. Fetch the transcript using your tool.
2. Generate a concise Summary (3-5 sentences).
3. Identify 5 Key Concepts mentioned in the video.
4. Create a 3-question Multiple-Choice Quiz (with an answer key at the bottom) to test the user's knowledge.

Format your response clearly with markdown headings for each section.

Make sure to use the tool get_youtube_transcript() ALWAYS to fetch the transcript before attempting to summarize or extract concepts. 
NEVER fetch transcript yourself, instead, use the tool."""

# Create agent with the tool and inject the system prompt
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0) # Temp 0 reduces hallucinations for quizzes

# Inject the system instructions using the 'prompt' parameter
agent = create_react_agent(llm, [get_youtube_transcript], prompt=system_message)

# Test it with a natural URL prompt
result = agent.invoke({
    "messages": [("user", 
                  "Analyze this educational video: https://www.youtube.com/watch?v=dQw4w9WgXcQ")]
})
print("Video 1" ,result["messages"][-1].content)


result = agent.invoke({
    "messages": [("user", 
                  "Analyze this educational video: https://www.youtube.com/watch?v=0zvrGiPkVcs")]
})

print("Video 2" , result["messages"][-1].content)


result = agent.invoke({
    "messages": [("user", 
                  "Analyze this educational video: https://www.youtube.com/watch?v=xlX3VtuIfQ0")]
})

print("Video 3" , result["messages"][-1].content)

/var/folders/8f/xszcb9696l59gwdnxhr1fk_w0000gr/T/ipykernel_7775/2999270940.py:38: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, [get_youtube_transcript], prompt=system_message)


Fetching transcript for video ID: dQw4w9WgXcQ
Video 1 # Summary
The video features the iconic song "Never Gonna Give You Up" by Rick Astley. The lyrics express a deep commitment to love and loyalty, emphasizing the importance of trust and emotional connection in relationships. The singer reassures his partner that he will always be there for them, never abandoning or hurting them. The song has become a cultural phenomenon, often associated with the "Rickrolling" internet meme.

# Key Concepts
1. **Commitment in Relationships**: The importance of being dedicated and loyal to a partner.
2. **Emotional Connection**: The significance of understanding and sharing feelings in a relationship.
3. **Trust**: Building a foundation of trust to ensure a healthy relationship.
4. **Cultural Impact**: The song's influence on internet culture, particularly through the "Rickroll" meme.
5. **Reassurance**: The act of providing comfort and security to a loved one.

# Quiz
1. What is the main theme of the

# Acknowledgement

In [ ]:
# https://chatgpt.com/share/69a661a3-22c0-8012-9cf0-d1f0b80dc9d1
# https://gemini.google.com/share/37e2b1c4c39c